# Tema 1: Introducción a los Sistemas Agénticos

# 1.1 La Revolución Agéntica (2026)

## De Modelos de Lenguaje a Agentes Autónomos

### 📊 La Evolución del Paradigma

```
2018-2020: GPT-2/3        →  Generación de texto impresionante
2022-2023: ChatGPT        →  Conversación natural
2023-2024: Copilots       →  Asistencia en tareas específicas
2024-2026: Agents         →  Autonomía y ejecución de tareas complejas
```

### 🎯 ¿Qué Cambió?

**Antes (LLMs tradicionales):**
- Usuario escribe prompt → Modelo genera respuesta → Fin
- **Pasivo:** solo responde

**Ahora (Agentes):**
- Usuario define objetivo → Agente planifica → Agente ejecuta acciones → Agente verifica → Re-planifica si es necesario
- **Activo:** toma decisiones y acciones

In [ ]:
# Instalación de dependencias
!pip install openai yfinance python-dotenv pandas matplotlib -q

In [ ]:
import openai
import yfinance as yf
import json
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from typing import Dict, Any
import os
from google.colab import userdata


api_key = userdata.get('OPENAI_API_KEY')




### Ejemplo 1: LLM Tradicional (Pasivo)

Un LLM tradicional **no tiene acceso a herramientas** y solo puede responder con su conocimiento interno.

In [ ]:
import openai

def llm(user_query: str) -> str:
    """
    LLM clásico: recibe input, devuelve output, termina
    Sin acceso a herramientas, sin autonomía
    """

    client = openai.OpenAI(api_key=api_key)

    response = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": user_query}]
    )
    return response.choices[0].message.content

# Uso
query = "¿Cuál es el precio de las acciones de Apple hoy?"
response = llm(query)
print(response)

Lo siento, como soy una inteligencia artificial, no tengo la capacidad de proporcionar información en tiempo real o actualizaciones de noticias. Te recomendaría buscar el precio de intercambio de las acciones de Apple en tiempo real a través de una fuente financiera confiable como Google Finance, Yahoo Finance, Bloomberg, etc.


**❌ Limitaciones del LLM tradicional:**
- No puede acceder a datos reales
- No puede ejecutar acciones
- Usuario debe hacer todo manualmente

### Ejemplo 2: Agente con Tools (Activo)

Un agente **puede usar herramientas** para acceder a datos reales y ejecutar acciones.

In [ ]:
#definir el schema de la función

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": "Obtiene el precio actual de una acción por su símbolo",
            "parameters": {
                "type": "object",
                "properties": {
                    "symbol": {
                        "type": "string",
                        "description": "Símbolo de la acción (ej: AAPL, GOOGL, TSLA)"
                    }
                },
                "required": ["symbol"]
            }
        }
    }
]

print(type(TOOLS))

<class 'list'>


In [ ]:
# Paso 2: Implementar la función real

def get_stock_price(symbol: str) -> Dict[str, Any]:
    """
    Herramienta real que obtiene precio de acciones desde Yahoo Finance
    """
    try:
        stock = yf.Ticker(symbol)
        data = stock.history(period="1d")

        if data.empty:
            return {"error": f"No se encontraron datos para {symbol}"}

        current_price = data['Close'].iloc[-1]
        prev_close = stock.info.get('previousClose', current_price)
        change = current_price - prev_close
        change_percent = (change / prev_close) * 100

        return {
            "symbol": symbol,
            "price": round(current_price, 2),
            "change": round(change, 2),
            "change_percent": round(change_percent, 2),
            "timestamp": datetime.now().isoformat()
        }
    except Exception as e:
        return {"error": str(e)}

In [ ]:
# Paso 3: Implementar el agente

def Agent(user_query: str, verbose: bool = True) -> str:
    """
    Agente que puede decidir autónomamente qué herramientas usar
    """
    messages = [{"role": "user", "content": user_query}]

    if verbose:
        print("🤖 Agente iniciado...")
        print(f"📝 Query: {user_query}\n")

    # Initialize the OpenAI client with the new API syntax
    client = openai.OpenAI(api_key=api_key)

    # Primera llamada: el modelo decide si necesita tools
    response = client.chat.completions.create(
        model="gpt-4",
        messages=messages,
        tools=TOOLS,
        tool_choice="auto"
    )

    response_message = response.choices[0].message

    # Si el modelo quiere usar herramientas
    if response_message.tool_calls:
        messages.append(response_message)

        # Ejecutar cada tool que el modelo solicitó
        for tool_call in response_message.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)

            if verbose:
                print(f"🔧 Usando herramienta: {function_name}")
                print(f"Argumentos: {function_args}")

            # Ejecutar la función
            if function_name == "get_stock_price":
                function_response = get_stock_price(**function_args)
            else:
                function_response = {"error": "Función no encontrada"}

            if verbose:
                print(f"   Resultado: {function_response}\n")

            # Agregar el resultado al contexto
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": json.dumps(function_response)
            })

        # Segunda llamada: el modelo genera respuesta final
        final_response = client.chat.completions.create(
            model="gpt-4",
            messages=messages
        )

        return final_response.choices[0].message.content
    else:
        # El modelo decidió no usar herramientas
        return response_message.content

In [ ]:
# Paso 4: Usar el agente

print("=" * 60)
print("EJEMPLO: Agente con Tools")
print("=" * 60)

query = "¿Cuál es el precio de las acciones de Apple hoy?"
response = Agent(query, verbose=True)

print("\n✅ Respuesta final:")
print(response)

EJEMPLO: Agente con Tools
🤖 Agente iniciado...
📝 Query: ¿Cuál es el precio de las acciones de Apple hoy?

🔧 Usando herramienta: get_stock_price
Argumentos: {'symbol': 'AAPL'}
   Resultado: {'symbol': 'AAPL', 'price': np.float64(273.05), 'change': np.float64(2.82), 'change_percent': np.float64(1.04), 'timestamp': '2026-04-20T20:13:40.238832'}


✅ Respuesta final:
El precio de las acciones de Apple hoy es de $273.05. Hubo un cambio de $2.82, lo que representa un aumento del 1.04%.


### 📊 Comparación Visual

| Característica | LLM Tradicional | Agente |
|---|---|---|
| Autonomía | ❌ Ninguna | ✅ Alta |
| Acceso a datos reales | ❌ No | ✅ Sí |
| Puede ejecutar acciones | ❌ No | ✅ Sí |
| Número de interacciones | 1 (single-shot) | N (multi-step) |
| Planning | ❌ No | ✅ Sí |
| Tools | ❌ 0 | ✅ Múltiples |

### 🔄 ChatGPT vs Copilots vs Agentes

| Dimensión | ChatGPT (LLM) | GitHub Copilot | Agente Verdadero |
|-----------|---------------|----------------|------------------|
| **Autonomía** | ❌ Cero (espera input) | ⚠️ Baja (sugiere) | ✅ Alta (ejecuta) |
| **Tools** | ❌ No tiene | ⚠️ Solo código | ✅ Múltiples |
| **Planning** | ❌ No | ❌ No | ✅ Sí |
| **Loops** | ❌ Single-shot | ❌ No | ✅ Sí (ReAct) |
| **State** | ❌ Stateless | ⚠️ Context file | ✅ State complejo |
| **Objetivo** | Responder | Asistir | Completar tarea |

## 📈 El "Agentic AI Moment": Por Qué Ahora

### Convergencia de 4 Factores

#### 1. **Models más capaces**
- GPT-4, Claude 3.5, Gemini 1.5: razonamiento avanzado
- Context windows largos (128K-1M+ tokens)
- Function calling nativo y robusto

#### 2. **Infraestructura madura**
- APIs estables y escalables
- Sandboxing seguro (E2B, Modal)
- Vector DBs producción-ready (Pinecone, Weaviate)

#### 3. **Frameworks especializados**
- LangChain, LangGraph (2023)
- AutoGen (Microsoft, 2023)
- CrewAI (2024)
- Swarm (OpenAI, 2024)

#### 4. **Casos de uso probados**
- Devin: coding autónomo (Cognition Labs)
- Claude Computer Use: control de GUI
- Copilot Workspace: flujos de desarrollo completos

## 🌍 Casos de Uso Transformadores en Producción

### 1. **Devin - Autonomous Software Engineer**
- **Qué hace:** Resuelve issues de GitHub end-to-end
- **Cómo:** Lee issue → planifica → escribe código → ejecuta tests → debuggea → hace commit
- **Impacto:** SWE-bench: 13.86% → 43.8% en 12 meses

### 2. **Claude Computer Use**
- **Qué hace:** Controla computadora como humano
- **Cómo:** Ve pantalla → decide acciones → mueve mouse, teclea, hace clicks
- **Status:** Beta pública (Oct 2024)

### 3. **GitHub Copilot Workspace**
- **Qué hace:** De issue → PR completo
- **Impacto:** Reduce tiempo de feature development 40-60%

### 4. **Customer Support Agents**
- **Qué hace:** Resuelve tickets automáticamente
- **Impacto:** Resolution rate 60-70% en tier-1

---
# 1.2 Definición Formal de Agente Inteligente

## ¿Qué es un Agente Inteligente?

### 📚 Definición Académica (Russell & Norvig)

> **Agente:** Entidad que percibe su **entorno** a través de **sensores** y actúa sobre ese entorno a través de **actuadores**

### 🎯 Definición para Agentes LLM (2026)

> **Agente Inteligente Basado en LLM:** Sistema que usa un modelo de lenguaje como motor de razonamiento para percibir información, tomar decisiones autónomas, y ejecutar acciones a través de herramientas para lograr objetivos definidos por el usuario.

In [ ]:
# Modelo conceptual de un agente LLM

class AgenteLLM:
    """
    Estructura básica de un agente basado en LLM
    """

    def __init__(self, llm, tools, memory):
        self.llm = llm              # Motor de razonamiento
        self.tools = tools          # Herramientas disponibles
        self.memory = memory        # Estado persistente
        self.objetive = None

    def percibe(self):
        """
        Sensores en agentes LLM:
        - Input del usuario
        - Resultados de tools
        - Estado de la memoria
        """
        return {
            "user_input": self.get_user_input(),
            "tool_results": self.get_latest_tool_results(),
            "memory": self.memory.get_relevant_context()
        }

    def think(self, perceptions):
        """
        Razonamiento con LLM:
        1. Construir prompt con percepciones
        2. LLM genera pensamiento (thought)
        3. LLM decide acción (action)
        """
        prompt = self.construir_prompt(perceptions)
        response_llm = self.llm.generate(prompt, tools=self.tools)
        return response_llm

    def act(self, decision_llm):
        """
        Actuadores en agentes LLM:
        - Ejecutar tools (APIs, código, búsquedas)
        - Actualizar memoria
        - Responder al usuario
        """
        if decision_llm.tool_call:
            result = self.ejecutar_tool(decision_llm.tool_call)
            self.memory.add(result)
            return result
        else:
            return decision_llm.response

    def execute(self, objetive):
        """
        Loop principal: Percepción → Razonamiento → Acción
        """
        self.objetive = objetive

        while not self.objetivo_completado():
            perceptions = self.percibe()
            decision = self.think(perceptions)
            resultado = self.act(decision)

        return self.generar_respuesta_final()

## 🧩 Propiedades Fundamentales de un Agente

### 1. **Autonomía**
- **Definición:** Opera sin intervención humana constante
- **Ejemplo:** Usuario da objetivo de alto nivel, agente decide todos los pasos intermedios

### 2. **Reactividad**
- **Definición:** Responde a cambios en el entorno
- **Ejemplo:** Monitorea precios de acciones y reacciona a cambios significativos

### 3. **Pro-actividad**
- **Definición:** Toma iniciativa para cumplir objetivos
- **Ejemplo:** Anticipa necesidades del usuario y sugiere acciones

### 4. **Social Ability**
- **Definición:** Interactúa con otros agentes y humanos
- **Ejemplo:** Coordina con otros agentes en un sistema multi-agente

## 🔄 Ciclo del Agente: Percepción → Razonamiento → Acción

```
┌─────────────────────────────────────┐
│        1. PERCEPCIÓN                │
│  • User input                       │
│  • Tool results                     │
│  • Memory/context                   │
└──────────────┬──────────────────────┘
               ↓
┌─────────────────────────────────────┐
│        2. RAZONAMIENTO              │
│  • Analizar situación               │
│  • Planificar próximo paso          │
│  • Decidir herramienta a usar       │
└──────────────┬──────────────────────┘
               ↓
┌─────────────────────────────────────┐
│        3. ACCIÓN                    │
│  • Ejecutar tool                    │
│  • Actualizar memoria               │
│  • Responder usuario                │
└──────────────┬──────────────────────┘
               ↓
               ┌──────────┐
               │ ¿Objetivo│
               │alcanzado?│
               └─┬────┬───┘
                 │    │
              NO │    │ SÍ
                 ↓    ↓
            (loop) (fin)
```

---
# 1.3 Taxonomía de Agentes

## Tipos de Agentes según su Arquitectura Interna

### 1. **Agentes Reactivos (Reflex Agents)**

**Características:**
- Respuesta directa a estímulos (reglas if-then)
- Sin modelo del mundo
- Sin memoria de estados previos
- Rápidos pero limitados

**Casos de uso:**
- Chatbots simples de FAQ
- Clasificación de intents
- Respuestas automáticas básicas

### 2. **Agentes Deliberativos (Model-Based)**

**Características:**
- Mantienen modelo del mundo
- Planificación explícita
- Razonamiento sobre consecuencias
- Más lentos pero más capaces

**Casos de uso:**
- Agentes de investigación
- Planificación de proyectos
- Análisis complejos

### 3. **Agentes Híbridos**

**Características:**
- Combinan reactividad y deliberación
- Reacción rápida + planificación cuando es necesario
- Balance entre velocidad y capacidad

**Ejemplo:** ReAct (Reasoning + Acting)
- Alterna entre razonar y actuar
- Reacciona rápido cuando puede
- Planifica cuando la situación lo requiere

### Criterios de Selección

**Elige Reactivo cuando:**
- ✅ Tarea simple y repetitiva
- ✅ Latencia crítica (<100ms)
- ✅ Reglas claras y fijas

**Elige Deliberativo cuando:**
- ✅ Tarea compleja con múltiples pasos
- ✅ Necesitas planificación upfront
- ✅ Razonamiento profundo requerido

**Elige Híbrido cuando:**
- ✅ Balance entre velocidad y capacidad
- ✅ Tareas variables (a veces simples, a veces complejas)
- ✅ Uso general (el más común en 2026)

---
# 1.4 El Modelo de Lenguaje como Motor de Razonamiento

## Pre-entrenamiento: Capacidades Emergentes

Los LLMs modernos desarrollan capacidades que **no fueron explícitamente programadas**:

- ✅ Razonamiento lógico
- ✅ Arithmetic básico
- ✅ Traducción entre idiomas
- ✅ Extracción de información
- ✅ Clasificación
- ✅ Generación de código

Estas capacidades **emergen** del entrenamiento en grandes cantidades de texto.

## Instruction-Tuning: De Base Model a Assistant

```
Base Model (GPT-4 base)
    ↓
Instruction Tuning (fine-tuning con ejemplos)
    ↓
Assistant Model (ChatGPT)
    ↓
RLHF (Reinforcement Learning from Human Feedback)
    ↓
Aligned Assistant (sigue instrucciones, rechaza contenido dañino)
```

## Reasoning Models en 2026

### 1. **OpenAI o1, o3**
- Chain-of-Thought interno (no visible)
- Extended reasoning time
- Test-time compute scaling
- Excelente en math, coding, reasoning complejo

### 2. **Anthropic Claude Extended Thinking**
- Similar a o1 pero con thinking **visible**
- Control sobre thinking budget
- Transparencia en el razonamiento

### 3. **DeepSeek-R1**
- RL-based reasoning
- Open-source
- Competitivo con o1
- Opción para self-hosting

## Test-time Compute vs Pre-training Compute

**Pre-training compute:**
- Gastado durante entrenamiento del modelo
- Fijo una vez el modelo está entrenado
- Ejemplo: GPT-4 usó ~10^25 FLOPs en entrenamiento

**Test-time compute:**
- Gastado durante inferencia (cuando usas el modelo)
- Variable según la tarea
- Reasoning models usan MÁS test-time compute para pensar más

```
Modelo Normal:      Pregunta → [Pensar 1s] → Respuesta
Reasoning Model:    Pregunta → [Pensar 30s] → Respuesta mejor
```

## Limitaciones Inherentes de LLMs como Agentes

### 1. **Alucinaciones**
- Generan información falsa pero plausible
- No tienen certeza sobre su conocimiento
- **Mitigación:** Usar tools para verificar hechos

### 2. **Falta de Grounding**
- No conectados directamente con la realidad
- No pueden ver, oír, o sentir
- **Mitigación:** Tools para percepción (vision models, APIs)

### 3. **Ausencia de Memoria Nativa**
- Stateless: no recuerdan conversaciones previas
- Context window limitado
- **Mitigación:** Sistemas de memoria externos (RAG, bases de datos)

### 4. **Incapacidad de Acción Sin Tools**
- Solo pueden generar texto
- No pueden navegar web, ejecutar código, enviar emails nativamente
- **Mitigación:** Function calling y tool integration

### 5. **Context Window Limitado**
- Aunque grande (128K-1M tokens), sigue siendo finito
- Costo crece con contexto
- **Mitigación:** Summarization, context management

---
# 1.5 Estado del Arte 2026

## Benchmarks Principales

### 1. **SWE-bench (Software Engineering)**
- **Qué mide:** Capacidad de resolver issues reales de GitHub
- **Dataset:** 2,294 issues de Python repos populares
- **Evaluación:** Tests automatizados
- **SOTA (2026):** ~43% resolution rate

### 2. **WebArena (Web Navigation)**
- **Qué mide:** Navegación y ejecución de tareas en websites
- **Tareas:** E-commerce, forums, maps, etc.
- **SOTA (2026):** ~35% success rate

### 3. **OSWorld (Computer Use)**
- **Qué mide:** Control de sistema operativo completo
- **Tareas:** File management, app control, multi-app workflows
- **SOTA (2026):** ~22% success rate

### 4. **GAIA (General AI Assistants)**
- **Qué mide:** Tareas generales multi-modal
- **Incluye:** Razonamiento, búsqueda, análisis de imágenes
- **SOTA (2026):** ~40% en nivel avanzado

## Productos Comerciales Destacados

### 1. **Anthropic Claude Computer Use**
- Control de computadora via screenshots + actions
- Beta pública desde Oct 2024
- Casos de uso: Testing, automation, research

### 2. **Devin (Cognition Labs)**
- Autonomous software engineer
- End-to-end: issue → code → tests → PR
- Usado internamente en producción

### 3. **GitHub Copilot Workspace**
- De issue a PR completo
- Planificación + implementación + tests
- Limited preview para empresas

### 4. **Cursor AI**
- IDE con agente integrado
- Multi-file editing, debugging
- >100K desarrolladores activos

## Papers Fundamentales Recientes

### Must-Read Papers (2023-2025):

1. **ReAct: Synergizing Reasoning and Acting in LLMs** (Yao et al., 2023)
   - Patrón fundamental: Thought → Action → Observation

2. **Reflexion: Language Agents with Verbal RL** (Shinn et al., 2023)
   - Self-reflection y learning from mistakes

3. **AutoGen: Enabling Next-Gen LLM Applications** (Wu et al., 2023)
   - Framework multi-agente de Microsoft

4. **Tree of Thoughts** (Yao et al., 2023)
   - Exploración de múltiples caminos de razonamiento

5. **Gorilla: LLM Connected with Massive APIs** (Patil et al., 2023)
   - Tool use a gran escala

6. **HuggingGPT: Solving AI Tasks with ChatGPT** (Shen et al., 2023)
   - LLM como controlador de otros modelos

7. **MetaGPT: Meta Programming for Multi-Agent** (Hong et al., 2023)
   - Role-based multi-agent collaboration

---
# 📝 Resumen del Tema 1

## Puntos Clave

### 1.1 La Revolución Agéntica
- ✅ Agentes son **cualitativamente diferentes** de LLMs tradicionales
- ✅ **Autonomía** es la característica definitoria
- ✅ **Tools** permiten interactuar con el mundo real
- ✅ Convergencia de 4 factores: models, infra, frameworks, casos de uso

### 1.2 Definición Formal
- ✅ Agente = Percepción + Razonamiento + Acción
- ✅ LLM como motor de razonamiento
- ✅ Ciclo continuo hasta alcanzar objetivo

### 1.3 Taxonomía
- ✅ Reactivos: rápidos pero limitados
- ✅ Deliberativos: planificación explícita
- ✅ Híbridos (ReAct): balance óptimo

### 1.4 LLMs como Motores
- ✅ Capacidades emergentes del pre-training
- ✅ Reasoning models: o1, Claude, DeepSeek-R1
- ✅ Limitaciones: alucinaciones, falta de grounding, memoria

### 1.5 Estado del Arte
- ✅ Benchmarks: SWE-bench, WebArena, OSWorld, GAIA
- ✅ Productos: Claude Computer Use, Devin, Copilot Workspace
- ✅ Research activo con mejoras constantes

## Próximo Tema

**Tema 2: Componentes Fundamentales de un Agente**
- Agent Core (LLM selection)
- Sistema de Memoria
- Interfaz de Herramientas
- Módulo de Planificación
- Orquestación y Control

---
# 💪 Ejercicios Prácticos

## Ejercicio 1: Comparar LLM vs Agente

**Tarea:** Modifica el código del agente simple para que pueda:
1. Obtener precios de múltiples acciones
2. Compararlas
3. Recomendar cuál comprar basado en el cambio porcentual

## Ejercicio 2: Extender Herramientas

**Tarea:** Añade una nueva tool `get_company_news` que busque noticias recientes de una empresa usando una API de noticias.

## Ejercicio 3: Medir Autonomía

**Tarea:** Para una tarea específica (ej: "Analizar rentabilidad de mi cartera"), cuenta cuántos pasos intermedios:
- Un usuario debe hacer manualmente con un LLM tradicional
- Un agente puede hacer autónomamente

Calcula el % de autonomía.

In [ ]:
def comparar_acciones(actions_list):
 actions = {
     "AAPL": "buena",
     "GOOGL": "mala",
     "MSFT": "regular",
     "IBEX": "buenas"
    }
 return actions

In [ ]:
tool = [
    {
        "type": "function",
        "function": {
            "name": "comparar_acciones",
            "description": "Compara acciones",
            "parameters": {
                "type": "object",
                "properties": {
                    "symbol": {
                        "type": "string",
                        "description": "Símbolo de la acción (ej: AAPL, GOOGL, TSLA)"
                    }
                },
                "required": ["symbol"]
            }
        }
    }
]

print(type(tool))

<class 'list'>


---
# 📚 Referencias y Recursos Adicionales

## Papers
- ReAct: https://arxiv.org/abs/2210.03629
- Reflexion: https://arxiv.org/abs/2303.11366
- AutoGen: https://arxiv.org/abs/2308.08155

## Documentación
- OpenAI Function Calling: https://platform.openai.com/docs/guides/function-calling
- Anthropic Claude: https://docs.anthropic.com/
- LangChain: https://python.langchain.com/

## Blogs
- Lilian Weng - LLM Powered Autonomous Agents: https://lilianweng.github.io/posts/2023-06-23-agent/
- Anthropic - Building Effective Agents: https://www.anthropic.com/research

## Videos
- Andrew Ng - AI Agentic Design Patterns: https://www.deeplearning.ai/
